# NeuroRL GRPO Training — Google Colab (T4)

Fine-tunes **Qwen3-1.7B-Instruct** (Unsloth 4-bit + LoRA r=16) to decode motor
intents from a synthetic BCI environment using Group Relative Policy Optimisation (GRPO).

| | |
|---|---|
| **Target hardware** | Google Colab — single T4 GPU (free tier OK) |
| **Environment server** | `https://abhishekbiradar-neuro-rl-env.hf.space` |
| **Adapter published to** | `$HF_USER/neuro-rl-adapter` (HF Hub) |
| **Training steps** | 300 |

## Before you run

1. **Runtime → Change runtime type → T4 GPU**.
2. Open the **🔑 Secrets** sidebar and add three secrets (toggle *Notebook access* ON for each):
   - `WANDB_API_KEY` — from https://wandb.ai/authorize
   - `HF_TOKEN` — write-scope token from https://huggingface.co/settings/tokens
   - `HF_USER` — your HuggingFace username (target repo: `$HF_USER/neuro-rl-adapter`)
3. Run cells top-to-bottom. If Cell 2 throws an `ImportError`, **Runtime → Restart session** and re-run from Cell 2.

In [ ]:
import subprocess, os, sys

# ── Upgrade unsloth + unsloth_zoo together ───────────────────────────────
# Colab's preinstalled trl/unsloth combo can drift; the well-known failure
# mode is unsloth_zoo importing `ConstantLengthDataset` from a TRL path that
# was removed in TRL >= 1.0.  Upgrading both packages together avoids it.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "unsloth", "unsloth_zoo"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "trl", "transformers", "accelerate", "peft", "datasets", "wandb", "uv"],
    check=True,
)
# neuro_rl_env from the deployed HuggingFace Space.
# uv handles HF Spaces git better than pip's --filter=blob:none partial-clone.
subprocess.run(
    ["uv", "pip", "install", "--system", "-q",
     "git+https://huggingface.co/spaces/abhishekBiradar/neuro-rl-env"],
    check=True,
)

# ── Colab Secrets → environment variables ────────────────────────────────
from google.colab import userdata

for _key in ("WANDB_API_KEY", "HF_TOKEN", "HF_USER"):
    try:
        os.environ[_key] = userdata.get(_key)
    except Exception:
        print(f"⚠  Secret {_key} not set — add it in the 🔑 sidebar.")

# HF login (writes to ~/.huggingface so push_to_hub works later).
if os.environ.get("HF_TOKEN"):
    subprocess.run(
        ["huggingface-cli", "login", "--token", os.environ["HF_TOKEN"],
         "--add-to-git-credential"],
        check=False,
    )

print("All packages installed.")
print("⚠  If the next cell raises ImportError, Runtime → Restart session, then re-run from Cell 2.")

In [ ]:
import json, re, torch
from neuro_rl_env import NeuroRLClient
from neuro_rl_env.models import NeuroRLAction, INTENTS

BASE_URL = os.environ.get("NEURO_RL_URL", "https://abhishekbiradar-neuro-rl-env.hf.space")
USE_GPU  = torch.cuda.is_available()

print(f"BASE_URL : {BASE_URL}")
print(f"USE_GPU  : {USE_GPU}  ({torch.cuda.get_device_name(0) if USE_GPU else 'CPU'})")
print(f"INTENTS  : {INTENTS}")

assert USE_GPU, "Switch Runtime → Change runtime type → T4 GPU before continuing."

In [ ]:
# ── §12 Constraint 10 ──────────────────────────────────────────────────────
# use_vllm is left at its default (False).  Qwen3 uses a non-standard
# KV-cache layout that Unsloth's vLLM backend mis-handles, returning
# incorrect log-probs and causing GRPO advantage estimates to diverge.
# Re-enable only after upstream Unsloth/vLLM alignment is confirmed.
# ───────────────────────────────────────────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth",
)
print("Loaded Qwen3-1.7B (4-bit) with LoRA r=16")

In [ ]:
SYSTEM_PROMPT = """\
You are a neural decoder for a brain-computer interface.

You receive mean firing rates (Hz) from 20 cortical neurons plus metadata.

Step 1 — reason inside <think>…</think>:
  • Which neurons fire significantly above baseline (>80 Hz)?
  • Which are suppressed (<25 Hz)?
  • Map the pattern to one motor intent.

Step 2 — output ONLY a JSON object (no other text after </think>):
{
  "intent":          one of ["move_left","move_right","move_up","move_down","grasp","release","rest"],
  "confidence":      float in [0.0, 1.0],
  "reasoning":       one-sentence justification,
  "signal_features": list of strings like "ch2_power", "ch7_power"
}"""


def make_prompt(obs_dict: dict) -> str:
    mfr = [round(r, 1) for r in obs_dict["mean_firing_rates"]]
    drift = obs_dict.get("drift_phase", 0.0)
    noise = obs_dict.get("noise_level", 0.0)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Mean firing rates (Hz, neurons 0-19): {mfr}\n"
        f"Drift phase: {drift:.3f}   Noise level: {noise:.1f} Hz\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


with NeuroRLClient(BASE_URL).sync() as _env:
    _obs = _env.reset()
_obs_dict = _obs.model_dump() if hasattr(_obs, "model_dump") else _obs.__dict__
print("─── Sample prompt (first 400 chars) ───")
print(make_prompt(_obs_dict)[:400])

In [ ]:
def _parse_action(text: str) -> NeuroRLAction:
    """Parse a NeuroRLAction from a model completion; fall back to 'rest'."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    m = re.search(r"\{.*?\}", text, re.DOTALL)
    if m:
        try:
            return NeuroRLAction.model_validate(json.loads(m.group()))
        except Exception:
            pass
    return NeuroRLAction.model_validate(
        {"intent": "rest", "confidence": 0.0, "reasoning": "parse_failed", "signal_features": []}
    )


def openenv_reward(prompts, completions, **kwargs):
    """GRPO reward function — one HTTP round-trip per completion."""
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[-1]["content"]
        action = _parse_action(text)
        with NeuroRLClient(BASE_URL).sync() as env:
            env.reset()
            obs = env.step(action)
        rewards.append(float(obs.reward))
    return rewards


_test_rewards = openenv_reward(
    prompts=["test"],
    completions=['<think>Neurons 0-4 are elevated.</think>\n{"intent":"move_left","confidence":0.8,"reasoning":"high ch0-4","signal_features":["ch0_power","ch2_power"]}'],
)
print(f"Reward function test reward: {_test_rewards[0]:+.4f}")

In [ ]:
# 63 prompts = 7 intents × 3 noise levels × 3 drift phases.
from datasets import Dataset

_NOISE_LEVELS = [5.0, 12.5, 20.0]
_DRIFT_PHASES = [0.0, 1.0, 2.0]

rows = []
with NeuroRLClient(BASE_URL).sync() as _env:
    for _intent in INTENTS:
        for _noise in _NOISE_LEVELS:
            for _drift in _DRIFT_PHASES:
                _obs = _env.reset()
                _obs_dict = _obs.model_dump() if hasattr(_obs, "model_dump") else _obs.__dict__
                rows.append({"prompt": make_prompt(_obs_dict)})

dataset = Dataset.from_list(rows)
print(f"Dataset: {len(dataset)} rows, columns: {dataset.column_names}")
print(dataset[0]["prompt"][:300])

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# Single T4 has 16 GB — keep per-device batch at 1 and lean on grad accum.
# T4 is Turing, which has no native bf16; use fp16 instead.
config = GRPOConfig(
    output_dir="/content/neuro-rl-agent",
    # ── Generation ──────────────────────────────────────────
    num_generations=4,
    max_completion_length=512,
    temperature=0.9,
    # ── Optimisation ────────────────────────────────────────
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    fp16=True,                  # T4 = Turing → fp16 (no native bf16)
    max_steps=300,              # >= 3 drift cycles of training
    # ── Logging / saving ────────────────────────────────────
    report_to=["wandb"],
    logging_steps=5,
    save_strategy="steps",
    save_steps=100,
    # use_vllm=False (default) — see Cell 3 note.
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[openenv_reward],
    args=config,
    train_dataset=dataset,
)

print(f"GRPOTrainer ready  max_steps={config.max_steps}  "
      f"num_generations={config.num_generations}  "
      f"batch={config.per_device_train_batch_size}  fp16={config.fp16}")

In [ ]:
import wandb

wandb.init(
    project="neuro-rl",
    name="colab_t4_run",
    config={
        "model": "Qwen3-1.7B",
        "max_steps": config.max_steps,
        "num_generations": config.num_generations,
        "learning_rate": config.learning_rate,
        "fp16": config.fp16,
        "env_url": BASE_URL,
    },
)

print(f"WandB run: {wandb.run.name}  url: {wandb.run.get_url()}")
print("Starting training …")

trainer.train()

wandb.finish()
print("Training complete.")

In [ ]:
"""Save artefacts to /content and push adapter to HF Hub."""
import pathlib, matplotlib.pyplot as plt

WORK = pathlib.Path("/content")

# ── (a) Save PEFT adapter ────────────────────────────────────────────────
ADAPTER_DIR = WORK / "adapter"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"Adapter saved to {ADAPTER_DIR}")

# ── (b) Push adapter to HF Hub: $HF_USER/neuro-rl-adapter ────────────────
HF_USER = os.environ.get("HF_USER", "abhishekBiradar")
HF_REPO = f"{HF_USER}/neuro-rl-adapter"
model.push_to_hub(HF_REPO, private=False)
tokenizer.push_to_hub(HF_REPO, private=False)
print(f"Adapter pushed to https://huggingface.co/{HF_REPO}")

# ── (c) Save WandB run URL ───────────────────────────────────────────────
_wandb_url = wandb.run.get_url() if wandb.run is not None else "wandb offline — no URL"
(WORK / "wandb_run.txt").write_text(_wandb_url)
print(f"WandB URL saved: {_wandb_url}")

# ── (d) Reward and loss curves ───────────────────────────────────────────
_log_history = trainer.state.log_history

def _extract(*keys):
    for key in keys:
        rows = [(e["step"], e[key]) for e in _log_history if key in e]
        if rows:
            xs, ys = zip(*rows)
            return list(xs), list(ys)
    return [], []

fig, (ax_r, ax_l) = plt.subplots(1, 2, figsize=(12, 4))

steps_r, rewards = _extract("reward", "train/reward", "rewards/mean")
ax_r.plot(steps_r, rewards, linewidth=1.5)
ax_r.set_xlabel("step")
ax_r.set_ylabel("reward")
ax_r.set_title("GRPO Reward Curve")
ax_r.grid(True, alpha=0.3)

steps_l, losses = _extract("loss", "train/loss")
ax_l.plot(steps_l, losses, color="tab:orange", linewidth=1.5)
ax_l.set_xlabel("step")
ax_l.set_ylabel("loss")
ax_l.set_title("Training Loss Curve")
ax_l.grid(True, alpha=0.3)

fig.tight_layout()
_curve_path = WORK / "training_curves.png"
fig.savefig(str(_curve_path), dpi=150)
plt.show()
print(f"Curves saved to {_curve_path}")
print("All artefacts saved.")